# Exercise 2: Finding the Most Polarizing Movies Using Spark SQL (Python)
**Course:** 5AXD57 Big Data and Cloud Computing · **Student:** Ethan Le Chevallier

**Dataset:** MovieLens.

`u.data` = user ratings (user ID, movie ID, rating, timestamp).

`u.item` = movie information (movie ID, title, release date, genres).

**Objective:** identify movies that split viewers, i.e. movies with many very low (1 star) **and** many very high (5 stars) ratings, which often leads to a moderate average.

**This notebook** solves the exercise with PySpark DataFrames, as shown in the course (`spark.read.csv`, `toDF`, `join`, `groupBy().agg()`, `filter()`, `orderBy()`). The same analysis written in SQL gives the same result.

**Plan**

0. Load and check the data
1. Task 1: statistics per movie (average, total, 1-star, 5-star)
2. Task 2: keep the polarizing movies (≥ 100 ratings, ≥ 20 one-star, ≥ 20 five-star)
3. Task 3: polarization score and ranking
4. Task 4: final display
5. The final query and its output
6. Deliverable: explanation, then challenge question

## 0. Load and check the data
The two files are in a Unity Catalog Volume. As in the course, `spark.read.csv` is used with `sep` (the separator) and `inferSchema=True` (Spark detects the type of each column). The files have no header, so the columns are named with `toDF` for `u.data`, and with `select` + `alias` for `u.item` (only the id and the title are needed).

In [0]:
from pyspark.sql import functions as F

# u.data: tab-separated, no header -> name the 4 columns with toDF
ratings = spark.read.csv(
    "/Volumes/movielens/default/movielens/u.data",
    sep="\t", inferSchema=True, header=False
).toDF("user_id", "movie_id", "rating", "timestamp")

# u.item: pipe-separated, ISO-8859-1 encoding -> keep only the movie id and the title
movies = (
    spark.read.csv(
        "/Volumes/movielens/default/movielens/u.item",
        sep="|", inferSchema=True, header=False, encoding="ISO-8859-1"
    )
    .select(F.col("_c0").alias("movie_id"), F.col("_c1").alias("title"))
)

In [0]:
# Check: expected 100,000 ratings, 1,682 movies, ratings between 1 and 5
print("ratings:", ratings.count())
print("movies :", movies.count())
display(ratings.agg(F.min("rating").alias("min_rating"), F.max("rating").alias("max_rating")))

ratings: 100000
movies : 1682


min_rating,max_rating
1,5


## Task 1: Calculate movie statistics
**Approach:** join ratings and movies, group by movie, then compute four aggregates in a single pass with `agg`.
- `F.avg("rating")`: average rating
- `F.count("*")`: total number of ratings
- `F.count(F.when(F.col("rating") == 1, 1))` and the same with 5: conditional count. `F.when` returns a value only when the condition is true (otherwise null), and `F.count` ignores nulls, so only the 1-star (or 5-star) ratings are counted.

The result is kept in the DataFrame `movie_stats`, so Tasks 2 to 4 reuse it without recomputing the logic.

In [0]:
movie_stats = (
    ratings.join(movies, "movie_id", "inner")          # link both files with the movie identifier
    .groupBy("movie_id", "title")                      # one row per movie
    .agg(
        F.avg("rating").alias("avg_rating"),                                   # average rating
        F.count("*").alias("nb_ratings"),                                      # total number of ratings
        F.count(F.when(F.col("rating") == 1, 1)).alias("nb_1_star"),           # number of 1-star ratings
        F.count(F.when(F.col("rating") == 5, 1)).alias("nb_5_star"),           # number of 5-star ratings
    )
)

In [0]:
# Task 1 result: statistics for every movie (the 10 most rated are shown here)
display(
    movie_stats
    .select("movie_id", "title", F.round("avg_rating", 2).alias("avg_rating"),   # keep only 2 digits after the comma
            "nb_ratings", "nb_1_star", "nb_5_star")
    .orderBy(F.col("nb_ratings").desc())                                         # most rated one at the top
    .limit(10)                                                                   # show the 10 most rated
)

movie_id,title,avg_rating,nb_ratings,nb_1_star,nb_5_star
50,Star Wars (1977),4.36,583,9,325
258,Contact (1997),3.8,509,12,132
100,Fargo (1996),4.16,508,14,227
181,Return of the Jedi (1983),4.01,507,8,171
294,Liar Liar (1997),3.16,485,47,46
286,"English Patient, The (1996)",3.66,481,30,131
288,Scream (1996),3.44,478,32,85
1,Toy Story (1995),3.88,452,8,119
300,Air Force One (1997),3.63,431,14,85
121,Independence Day (ID4) (1996),3.44,429,22,76


## Task 2: Identify polarizing movies
**Approach:** keep only the movies that satisfy all three conditions with `filter()`. With several conditions, each one is in parentheses and they are combined with `&` (AND), as in the course. The result is kept in the DataFrame `polarizing_movies` to be used in Task 3 and in the final display.

We need to satisfy those 3 conditions:
-  At least 100 ratings.
-  At least 20 ratings of 1 star.
-  At least 20 ratings of 5 stars.

In [0]:
polarizing_movies = movie_stats.filter(
    (F.col("nb_ratings") >= 100)      # enough ratings to be representative
    & (F.col("nb_1_star") >= 20)      # at least 20 one-star ratings
    & (F.col("nb_5_star") >= 20)      # at least 20 five-star ratings
)

In [0]:
# Number of movies kept out of 1,682
print("Number of polarizing movies:", polarizing_movies.count())

Number of polarizing movies: 11


We only have 11 polarizing movies in the dataset.

## Task 3: Rank the results
**Polarization score = number of 1-star ratings + number of 5-star ratings.**

The score is computed as a new column and the movies are sorted in descending order (ties broken by `movie_id` so the order is stable).

In [0]:
display(
    polarizing_movies
    .select("title", (F.col("nb_1_star") + F.col("nb_5_star")).alias("polarization_score"))   # Task 3 formula
    .orderBy(F.col("polarization_score").desc(), "movie_id")                                  # in case of ties order by movie ID
)

title,polarization_score
Pulp Fiction (1994),209
"English Patient, The (1996)",161
Scream (1996),117
Leaving Las Vegas (1995),105
Independence Day (ID4) (1996),98
Liar Liar (1997),93
Evita (1996),69
"Saint, The (1997)",64
Twister (1996),55
Mars Attacks! (1996),54


## Task 4: Display the results
Final query with the six requested columns.

**Movie title / Average rating / Total ratings / 1-star ratings / 5-star ratings / Polarization score**

In [0]:
display(
    polarizing_movies
    .select(
        F.col("title").alias("movie_title"),
        F.round("avg_rating", 2).alias("average_rating"),
        F.col("nb_ratings").alias("total_ratings"),
        F.col("nb_1_star").alias("one_star_ratings"),
        F.col("nb_5_star").alias("five_star_ratings"),
        (F.col("nb_1_star") + F.col("nb_5_star")).alias("polarization_score"),
    )
    .orderBy(F.col("polarization_score").desc(), "movie_id")
)

movie_title,average_rating,total_ratings,one_star_ratings,five_star_ratings,polarization_score
Pulp Fiction (1994),4.06,394,21,188,209
"English Patient, The (1996)",3.66,481,30,131,161
Scream (1996),3.44,478,32,85,117
Leaving Las Vegas (1995),3.7,298,20,85,105
Independence Day (ID4) (1996),3.44,429,22,76,98
Liar Liar (1997),3.16,485,47,46,93
Evita (1996),2.98,259,39,30,69
"Saint, The (1997)",3.12,316,30,34,64
Twister (1996),3.22,293,20,35,55
Mars Attacks! (1996),2.85,217,34,20,54


## The final query (Tasks 1 to 4 in one chain) and the output of the results below

In [0]:
final_result = (
    ratings.join(movies, "movie_id", "inner")                                  # join
    .groupBy("movie_id", "title")                                              # groupBy
    .agg(F.avg("rating").alias("average_rating"),                              # agg
         F.count("*").alias("total_ratings"),
         F.count(F.when(F.col("rating") == 1, 1)).alias("one_star_ratings"),
         F.count(F.when(F.col("rating") == 5, 1)).alias("five_star_ratings"))
    .filter((F.col("total_ratings") >= 100)                                    # filter with &
            & (F.col("one_star_ratings") >= 20)
            & (F.col("five_star_ratings") >= 20))
    .withColumn("average_rating", F.round("average_rating", 2))
    .withColumn("polarization_score",
                F.col("one_star_ratings") + F.col("five_star_ratings"))
    .orderBy(F.col("polarization_score").desc(), "movie_id")                   # orderBy
    .select("title", "average_rating", "total_ratings",
            "one_star_ratings", "five_star_ratings", "polarization_score")
    .withColumnRenamed("title", "movie_title")
)
display(final_result)

movie_title,average_rating,total_ratings,one_star_ratings,five_star_ratings,polarization_score
Pulp Fiction (1994),4.06,394,21,188,209
"English Patient, The (1996)",3.66,481,30,131,161
Scream (1996),3.44,478,32,85,117
Leaving Las Vegas (1995),3.7,298,20,85,105
Independence Day (ID4) (1996),3.44,429,22,76,98
Liar Liar (1997),3.16,485,47,46,93
Evita (1996),2.98,259,39,30,69
"Saint, The (1997)",3.12,316,30,34,64
Twister (1996),3.22,293,20,35,55
Mars Attacks! (1996),2.85,217,34,20,54


### Same final query written in SQL with `spark.sql`
The two DataFrames are registered as temporary views, so the SQL query can read them.

In [0]:
ratings.createOrReplaceTempView("ratings")
movies.createOrReplaceTempView("movies")

display(spark.sql("""
WITH movie_stats AS (
  SELECT m.movie_id, m.title,
         AVG(r.rating)          AS avg_rating,
         COUNT(*)               AS nb_ratings,
         COUNT_IF(r.rating = 1) AS nb_1_star,
         COUNT_IF(r.rating = 5) AS nb_5_star
  FROM ratings r
  JOIN movies m ON r.movie_id = m.movie_id
  GROUP BY m.movie_id, m.title
)
SELECT title                    AS movie_title,
       ROUND(avg_rating, 2)     AS average_rating,
       nb_ratings               AS total_ratings,
       nb_1_star                AS one_star_ratings,
       nb_5_star                AS five_star_ratings,
       nb_1_star + nb_5_star    AS polarization_score
FROM movie_stats
WHERE nb_ratings >= 100 AND nb_1_star >= 20 AND nb_5_star >= 20
ORDER BY polarization_score DESC, movie_id
"""))

movie_title,average_rating,total_ratings,one_star_ratings,five_star_ratings,polarization_score
Pulp Fiction (1994),4.06,394,21,188,209
"English Patient, The (1996)",3.66,481,30,131,161
Scream (1996),3.44,478,32,85,117
Leaving Las Vegas (1995),3.7,298,20,85,105
Independence Day (ID4) (1996),3.44,429,22,76,98
Liar Liar (1997),3.16,485,47,46,93
Evita (1996),2.98,259,39,30,69
"Saint, The (1997)",3.12,316,30,34,64
Twister (1996),3.22,293,20,35,55
Mars Attacks! (1996),2.85,217,34,20,54


## Deliverable: why the highest-ranked movies might generate divided opinions
The reasons below are interpretations based on the numbers and on what these movies are. We can't interpret the users' motivations as the data does not contain it.

- **Pulp Fiction (1994), score 209, average 4.06.** It has the most extreme ratings, but the split is very unbalanced: 188 five-star against 21 one-star ratings. It is a strongly loved movie with a minority who reject it. Its violent content and dark humour make it a film that people either love or dislike, with little middle ground.
- **The English Patient (1996), score 161, average 3.66.** A long, slow romantic drama, widely praised and heavily awarded. This type of movie attracts viewers who love it and viewers who find it boring, which explains 131 five-star and 30 one-star ratings.
- **Scream (1996), Leaving Las Vegas (1995), Independence Day (1996), Liar Liar (1997).** These movies rely on a strong style or a specific taste. The same features that attract one audience repel another.
- **Evita, Mars Attacks!, The Saint, Twister, Starship Troopers.** Lower in the ranking, with averages between 2.8 and 3.2. Here the moderate average is the result of two opposed groups: this is the pattern described in the exercise (moderate average without agreement).

The top of the ranking is also made of very popular movies (around 400 to 500 ratings). A popular movie reaches a wider audience, so it collects more extreme ratings in both directions by volume alone. This is the limit of the score defined in Task 3, addressed in the challenge below.

## Challenge question: account for the total number of ratings
**Problem:** the raw score (1-star + 5-star) increases with popularity. A movie with 500 ratings gets a higher score than a movie with 100 ratings even if its audience is not more divided.

**Solution:** divide the score by the total number of ratings, to get the share of extreme ratings:

`polarization_rate = (nb_1_star + nb_5_star) / nb_ratings`

This rate is a percentage between 0 and 1, so it is comparable between a movie with 100 ratings and a movie with 500 ratings.

In [0]:
display(
    polarizing_movies
    .select(
        "title",
        "nb_ratings",
        (F.col("nb_1_star") + F.col("nb_5_star")).alias("polarization_score"),                                   # raw score (Task 3)
        F.round((F.col("nb_1_star") + F.col("nb_5_star")) / F.col("nb_ratings"), 3).alias("polarization_rate"),  # normalised score
    )
    .orderBy(F.col("polarization_rate").desc())
)

title,nb_ratings,polarization_score,polarization_rate
Pulp Fiction (1994),394,209,0.53
Leaving Las Vegas (1995),298,105,0.352
"English Patient, The (1996)",481,161,0.335
Evita (1996),259,69,0.266
Mars Attacks! (1996),217,54,0.249
Scream (1996),478,117,0.245
Starship Troopers (1997),211,50,0.237
Independence Day (ID4) (1996),429,98,0.228
"Saint, The (1997)",316,64,0.203
Liar Liar (1997),485,93,0.192


**Reading the result**
- Pulp Fiction stays first: 53% of its ratings are either 1 or 5 stars.
- The order changes for the other movies. Leaving Las Vegas moves from 4th to 2nd, Evita from 7th to 4th and Mars Attacks! from 10th to 5th. Liar Liar, the most rated movie of the list (485 ratings), falls from 6th to 10th.
- The raw score favoured very popular movies. The rate removes this effect and shows which movies have the highest proportion of extreme opinions, whatever their popularity.